# PlantVillage — Pipeline complet (CR2 + CR4 + CR5) sur Colab

Notebook clé en main pour exécuter le projet de détection de maladies des plantes sur GPU gratuit Colab.

**Avant de lancer :**
1. **Runtime → Change runtime type → T4 GPU** (sinon entraînement très lent)
2. Préparez 2 fichiers sur Google Drive :
   - `Projet_Traitement_Images.zip` — le ZIP de votre projet local
   - `plantvillage-dataset.zip` — le dataset PlantVillage (téléchargeable depuis Kaggle)

**Estimation :** ~45-60 min de bout en bout (extraction + CR2 + CR4 + CR5).

## 1. Vérification du GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
assert gpus, "❌ Aucun GPU détecté. Runtime → Change runtime type → T4 GPU"
print(f"✅ TensorFlow {tf.__version__} — GPU(s) : {gpus}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Montage Google Drive et copie des ZIPs

Adaptez les chemins `DRIVE_PROJECT_ZIP` et `DRIVE_DATASET_ZIP` à l'emplacement de vos fichiers sur Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os
from pathlib import Path

# ⚠️ Adaptez ces chemins à votre Drive
DRIVE_PROJECT_ZIP = '/content/drive/MyDrive/Projet_Traitement_Images.zip'
DRIVE_DATASET_ZIP = '/content/drive/MyDrive/plantvillage-dataset.zip'

WORK = Path('/content/work')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

for src in (DRIVE_PROJECT_ZIP, DRIVE_DATASET_ZIP):
    assert Path(src).exists(), f'❌ Introuvable sur Drive : {src}'

shutil.copy(DRIVE_PROJECT_ZIP, WORK / 'Projet_Traitement_Images.zip')
print('✅ Projet copié')
shutil.copy(DRIVE_DATASET_ZIP, WORK / 'plantvillage-dataset.zip')
print('✅ Dataset copié')
!ls -lh /content/work

## 3. Extraction du projet + installation des dépendances

In [ ]:
%cd /content/work
!unzip -q -o Projet_Traitement_Images.zip
import os
# Si l'archive crée un sous-dossier, on s'y déplace
candidates = [d for d in os.listdir('.') if os.path.isdir(d) and 'Projet' in d]
PROJECT_DIR = candidates[0] if candidates else '.'
%cd $PROJECT_DIR
!ls

In [ ]:
!pip install -q -r requirements.txt
print('✅ Dépendances installées')

## 4. CR2 — Extraction du dataset + visualisations du prétraitement

`main.py` attend le ZIP dans le dossier courant. On le copie depuis `/content/work/`.

In [ ]:
import shutil
shutil.copy('/content/work/plantvillage-dataset.zip', '.')
!python main.py

In [ ]:
from IPython.display import Image, display
import os
for f in sorted(os.listdir('output_cr2')):
    if f.endswith('.png'):
        print(f'── {f}')
        display(Image(f'output_cr2/{f}'))

## 5. CR4 — Entraînement du CNN

**Durée attendue sur T4 :** ~1-2 min/époque × 30-50 époques (EarlyStopping) ≈ **30 min à 1 h**.

Output principal : `outputs/best_model.keras` + 5 figures + `history.json`.

In [ ]:
!python cr4_train.py

In [ ]:
from IPython.display import Image, display
for f in ['cr4_summary.png', 'cr4_loss_curves.png', 'cr4_accuracy_curves.png',
          'cr4_learning_rate.png', 'cr4_train_val_gap.png']:
    p = f'outputs/{f}'
    if os.path.exists(p):
        print(f'── {f}')
        display(Image(p))

## 6. CR5 — Évaluation sur le jeu de test

Charge `best_model.keras` et produit : matrice de confusion, classification report, ROC, PR, scatter déséquilibre, exemples mal classifiés.

In [ ]:
!python cr5_evaluate.py

In [ ]:
from IPython.display import Image, display
for f in ['cr5_confusion_matrix.png', 'cr5_roc_curves.png', 'cr5_pr_curves.png',
          'cr5_misclassified.png', 'cr5_imbalance_vs_f1.png']:
    p = f'outputs/{f}'
    if os.path.exists(p):
        print(f'── {f}')
        display(Image(p))

In [ ]:
import json, pandas as pd
with open('outputs/cr5_summary.json') as f:
    print(json.dumps(json.load(f), indent=2))
print('\n── Classification report (extrait) ──')
df = pd.read_csv('outputs/cr5_classification_report.csv', index_col=0)
print(df.head(10).to_string())

## 7. Sauvegarde des résultats sur Google Drive

Toutes les sorties sont copiées dans `MyDrive/PlantVillage_results/` pour récupération hors Colab.
Le modèle `best_model.keras` (~150 Mo) est inclus.

In [ ]:
import shutil
from pathlib import Path
DEST = Path('/content/drive/MyDrive/PlantVillage_results')
DEST.mkdir(parents=True, exist_ok=True)
for src in Path('outputs').iterdir():
    shutil.copy(src, DEST / src.name)
for src in Path('output_cr2').iterdir():
    shutil.copy(src, DEST / src.name)
print(f'✅ Résultats copiés dans : {DEST}')
!ls -lh /content/drive/MyDrive/PlantVillage_results

## 8. (Optionnel) Téléchargement direct

Si vous préférez un ZIP téléchargé directement par le navigateur :

In [ ]:
import shutil
shutil.make_archive('/content/results', 'zip', 'outputs')
from google.colab import files
files.download('/content/results.zip')